# Deriving M20, Gini Coefficient, Asymmetry, and Smoothness Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.convolution import Gaussian2DKernel
import photutils
import statmorph

### Using the Statmorph package

To save time, instead of writing my own software, we can use some existing code called Statmorph to derive the morphological parameters.  [The package overview is here.](https://statmorph.readthedocs.io/en/latest/overview.html)

This package requires four things to run: an image stamp to work from, a weight map for that image stamp, a normalized image of the PSF that belongs with that image stamp, and a segmentation map isolating the desired object within the image stamp.  Additionally, you'll want to use a mask image to remove interloping sources if there are, e.g., stars or background galaxies overlapping with your object in projection.

For this tutorial, I'm going to do things crudely: it will be up to you to ensure you use the best segmentation map, PSF model, and weight map available for the HSC stamps.

In [ ]:
# Read in the masked image
imarr = fits.getdata('masked.fits')

# Make a separate mask image
mask = imarr == -999

#### Weight map

Here I'm making the simplest assumption: that all noise in this image is Poisson noise.  Hence, the weight map, defined in this way as 1/σ^2 (where σ^2 is the variance, which in Poisson statistics is the expectation value, which is just the number of counts in the pixel).  If HSC has weight maps available, use those.  If not, we'll want to think about the best way to derive a more appropriate one.

In [ ]:
weight = 1/np.abs(imarr)  # Negative weights mess things up
weight[weight == 1/999] = 1e-30  # Setting the mask values to a very small number (low weight)

#### PSF

Here I'm assuming the PSF is well-approximated by a Gaussian with a FWHM equal to some seeing (0.5" FWHM).  Some tests will follow at the end of this notebook on how sensitive the derived parameters are to the PSF model.

In [ ]:
# Just making a 2D Gaussian, assuming a seeing of 0.5 arcseconds
pxScale = 0.17
sigma = 2 * np.sqrt(2*np.log(2)) * 0.5  # Converting from FWHM
psf = Gaussian2DKernel(sigma/pxScale, sigma/pxScale).array  # This is normalized to 1 by default

In [ ]:
fig, ax = plt.subplots(1, figsize=(2, 2))
ax.imshow(psf)

#### Segmentation map

Here I'm creating this using Photutils, but a better option would probably be the HSC detection footprint, converted to the appropriate format.

In [ ]:
noisethresh = 0.00182 * 5  # 5sigma BG noise estimate from surface_photometry_demo.ipynb (sbLim)
npixels = 5  # minimum number of connected pixels
segm = photutils.detect_sources(imarr, noisethresh, npixels, mask=mask)

# Keep only the largest segment
label = np.argmax(segm.areas) + 1
segmap = segm.data == label

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(imarr, vmin=-0.1, vmax=0.9)
ax[0].set_title('Image stamp')
ax[1].imshow(segmap)
ax[1].set_title('Segmentation map')

### Deriving the parameters

Once you have everything in hand, running the software is trivial.

In [ ]:
source_morphs = statmorph.source_morphology(
    imarr,
    segmap,
    weightmap=weight,
    psf=psf,
    mask=mask,)
morph = source_morphs[0]

In [ ]:
print('M20: %.4f' % (morph.m20))
print('Gini: %.4f' % (morph.gini))
print('Smoothness: %.4f' % (morph.smoothness))
print('Asymmetry: %.4f' % (morph.asymmetry))

This also derives tons of other things, including some we derived ourselves in the surface_photometry_demo.ipynb notebook.  It doesn't do them in the same way, so the values will be different, but we can use them as a sanity check that this package isn't doing anything outrageous.

[All available parameters are listed here.](https://statmorph.readthedocs.io/en/latest/api.html)

In [ ]:
print('Half-light radius: %.4f arcsec' % (morph.rhalf_ellip*pxScale))
print('Petrosian radius: %.4f arcsec' % (morph.rpetro_ellip*pxScale))
print('Concentration: %.4f' % (morph.concentration))

To compare, our values from surface photometry were:
   - Half-light radius = 4.7118 arcsec
   - C82 = 2.4703
   - Petrosian radius = 9.0045 arcsec
   
Which are all pretty close.

### PSF sensitivity

Here I'm testing how sensitive these parameters are to the PSF by re-running the software with the same mask, segmentation map, and weight map, but varying the PSF model used.

In [ ]:
def mkPsf(seeing):
    sigma = 2 * np.sqrt(2*np.log(2)) * seeing
    psf = Gaussian2DKernel(sigma/pxScale, sigma/pxScale).array
    
    return psf

seeings = np.linspace(0.1, 2, 10)  # An array of seeing values in arcsec

In [ ]:
store = []
for i in range(len(seeings)):
    psf = mkPsf(seeings[i])
    source_morphs = statmorph.source_morphology(
        imarr,
        segmap,
        weightmap=weight,
        psf=psf,
        mask=mask,)
    morph = source_morphs[0]
    store.append(morph)

In [ ]:
ginis = [i.gini for i in store]
m20s = [i.m20 for i in store]
asyms = [i.asymmetry for i in store]
smths = [i.smoothness for i in store]

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12, 3))
ax[0].plot(seeings, ginis, 'k.', label='Gini')
ax[0].legend()
ax[1].plot(seeings, m20s, 'rs', label='M20')
ax[1].legend()
ax[2].plot(seeings, asyms, 'b^', label='Asymmetry')
ax[2].legend()
ax[3].plot(seeings, smths, 'go', label='Smoothness')
ax[3].legend()
plt.tight_layout()

Good news: it doesn't change the results at all.  In fact... looking at the [source code](https://statmorph.readthedocs.io/en/latest/_modules/statmorph/statmorph.html), it seems that the PSF model is only used for Sersic fitting.  So if we want Sersic parameters we'll need to be concerned with this, but if we don't, or if we'd rather use GALFIT or some other software to derive those, then it just needs a PSF because it by default fits a Sersic profile.  Doesn't concern the parameters we want.